# Knowledge Distillation: Llama-3.1-8B → Llama-3.2-1B
**CS 455 Term Project** — Ali Kumral, Revna Demirkale

---
## ⚠️ MUST RUN FIRST — run the cell below after every runtime restart
Everything else in this notebook depends on it. It is safe to re-run at any time.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# MUST RUN FIRST — run after every runtime restart before anything else
# ═══════════════════════════════════════════════════════════════════════
import os, sys

# ── constants (change these if your paths differ) ──────────────────────
REPO_URL   = "https://github.com/alikumral/Knowledge-Distillation-Transferring-Mathematical-Reasoning"
REPO_NAME  = "Knowledge-Distillation-Transferring-Mathematical-Reasoning"
REPO_PATH  = f"/content/{REPO_NAME}"
DRIVE_HOME = "/content/drive/MyDrive/CS455/TermProject/Knowledge-Distillation-Transferring-Mathematical-Reasoning"

# ── 1. GPU memory allocator (prevents fragmentation OOM) ───────────────
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ── 2. Clone (first run) or pull latest (after restart) ────────────────
if not os.path.exists(REPO_PATH):
    print("Cloning repo...")
    os.system(f"git clone {REPO_URL}")
else:
    print("Repo exists, pulling latest changes...")
    os.system(f"git -C {REPO_PATH} pull")

# ── 3. cd + make mathdistill importable ────────────────────────────────
os.chdir(REPO_PATH)
SRC_PATH = os.path.join(REPO_PATH, "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# ── 4. Install Python deps (fast after first run; deps are pip-cached) ─
os.system("pip install -q -e . -r requirements.txt")

# ── 5. Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
os.environ["MATHDISTILL_HOME"] = DRIVE_HOME
os.makedirs(DRIVE_HOME, exist_ok=True)

# ── 6. Hugging Face login ──────────────────────────────────────────────
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)

# ── 7. Summary ─────────────────────────────────────────────────────────
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "⚠ NO GPU — switch runtime"
print(f"\n{'='*55}")
print(f"  GPU:       {gpu}")
print(f"  VRAM:      {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")
print(f"  Repo:      {os.getcwd()}")
print(f"  Artifacts: {os.environ['MATHDISTILL_HOME']}")
print(f"{'='*55}")
print("  Setup complete. Ready to run any stage below.")
print(f"{'='*55}")

---
## [OPTIONAL] Smoke test — only run once to confirm both models work
⚠️ **Do NOT run this before Stage 1 (generation).** It loads both models into VRAM and even after `del model` there is enough fragmentation to OOM the 8B generation. Skip this cell in normal use — it already passed.

In [ ]:
# OPTIONAL — already passed. Skip before running Stage 1.
import torch
from mathdistill.utils import load_config
from mathdistill.models import load_model_4bit, load_tokenizer, render_prompt
from mathdistill.prompts import build_messages
from mathdistill.answers import extract_pred_number

tcfg = load_config("configs/teacher_gen.yaml")
q = ("Natalia sold clips to 48 friends in April, and half as many in May. "
     "How many clips did she sell altogether?")

for model_id in [tcfg["model_id"], "meta-llama/Llama-3.2-1B-Instruct"]:
    print("=" * 60, "\nLoading", model_id)
    tok = load_tokenizer(model_id)
    model = load_model_4bit(model_id, tcfg["quantization"])
    enc = tok(render_prompt(tok, build_messages(q)), return_tensors="pt").to(model.device)
    out = model.generate(**enc, max_new_tokens=256, do_sample=False, pad_token_id=tok.pad_token_id)
    text = tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    print(text)
    print(">> extracted:", extract_pred_number(text), "(expected 72)")
    del model
    torch.cuda.empty_cache()

---
## Stage 1 — Teacher CoT Generation
Runs Llama-3.1-8B (4-bit) over the GSM8K train split, sampling `k=4` chain-of-thought traces per problem. Writes to Drive incrementally — **resumable** if the session drops.

**Pilot first** (`PILOT = True`): runs on 30 problems (~3 min) to validate throughput and VRAM before the full ~6h run. Check the log says `LIMIT set -> pilot mode: first 30 problems only`.

In [ ]:
from mathdistill.generate import generate_teacher_traces
from mathdistill.utils import load_config

PILOT = True   # ← set False for the full ~6h run once pilot passes

cfg = load_config("configs/teacher_gen.yaml")
cfg["batch_size"] = 4            # T4: effective batch = 4*k=16 seqs; safe for 14.5 GB VRAM
if PILOT:
    cfg["limit"] = 30            # ~3 min; validates throughput + VRAM

generate_teacher_traces(cfg, load_config("configs/data.yaml"))

---
## Stage 2 — Rejection Sampling & Build SFT Datasets
Filters teacher traces to those matching the gold answer, reports acceptance stats (the **Risk-1 check**), and writes the three SFT datasets: `gold`, `teacher_1`, `teacher_3`. CPU-only — no GPU needed.

In [ ]:
import json
from mathdistill.reject import rejection_sample, acceptance_stats, build_all_sft_datasets
from mathdistill.utils import artifact_path, load_config

traces_path   = artifact_path("traces", "teacher_traces.jsonl")
accepted_path = artifact_path("sft", "accepted.jsonl")

rejection_sample(traces_path, accepted_path)
stats = acceptance_stats(accepted_path)
print(json.dumps(stats, indent=2))

# Risk-1 check: teacher should solve ≥60% of problems
rate = stats["n_problems_with_ge1"] / max(1, stats.get("n_accepted_traces", 1))
if stats["n_problems_with_ge1"] < 18:   # <60% of 30-problem pilot
    print("\n⚠ Low acceptance — consider Plan B (more samples or swap teacher)")
else:
    print("\n✓ Acceptance rate looks good — safe to proceed")

# Only run this after the FULL generation (not pilot) to build the SFT datasets
# build_all_sft_datasets(accepted_path, load_config("configs/data.yaml"))

---
## Stage 3 — QLoRA Fine-tuning
Trains one (condition, seed) adapter. Change `CONFIG` and `SEED` per run. Spread runs across sessions to stay within Colab's GPU quota (~2h per run).

**Run pilot first** with a small subset to confirm loss decreases before a full 2h run.

In [ ]:
from mathdistill.train import train_one
from mathdistill.utils import load_config

# ── change these per run ───────────────────────────────────────────────
CONFIG = "configs/train_teacher_1.yaml"   # train_teacher_1 | train_teacher_3 | train_gold
SEED   = 0                                # 0, 1, 2
# ──────────────────────────────────────────────────────────────────────

train_one(load_config(CONFIG), load_config("configs/data.yaml"), seed=SEED)

---
## Stage 4 — Evaluation
Evaluates each condition (greedy decoding) on GSM8K test + MATH-200, reports accuracy with 95% bootstrap CIs.

In [ ]:
# Run the conditions you have trained. Comment out what you don't need yet.
STUDENT = "meta-llama/Llama-3.2-1B-Instruct"
TEACHER = "meta-llama/Llama-3.1-8B-Instruct"

!python scripts/run_eval.py --model-id {STUDENT} --name zeroshot
# !python scripts/run_eval.py --model-id {STUDENT} --adapter adapters/teacher_1/seed0 --name teacher_1_seed0
# !python scripts/run_eval.py --model-id {STUDENT} --adapter adapters/teacher_3/seed0 --name teacher_3_seed0
# !python scripts/run_eval.py --model-id {STUDENT} --adapter adapters/gold/seed0     --name gold_seed0
# !python scripts/run_eval.py --model-id {TEACHER} --name teacher

In [ ]:
# Aggregate all prediction files into a results table.
import glob, os, pandas as pd
from mathdistill.utils import read_jsonl
from mathdistill.metrics import accuracy, bootstrap_ci

pred_dir = os.path.join(os.environ["MATHDISTILL_HOME"], "results", "predictions")
rows = []
for path in sorted(glob.glob(os.path.join(pred_dir, "*.jsonl"))):
    data = read_jsonl(path)
    correct = [r["correct"] for r in data]
    lo, hi = bootstrap_ci(correct)
    rows.append({"condition": os.path.basename(path).replace(".jsonl", ""),
                 "n": len(correct), "acc": round(accuracy(correct), 4),
                 "ci_lo": round(lo, 4), "ci_hi": round(hi, 4)})
pd.DataFrame(rows)